# EDA — Ensino Fundamental

**Living Stone Foundation — Applied Data Lab**

**Audience:** non-authors of the project.  
**Level in focus:** **Ensino Fundamental** (`fundamental`).  
**Question this notebook answers:** What does official school abandonment look like, where is it higher, and which school characteristics move together with it?


## Variable dictionary (read before the charts)

| Column / label in charts | Plain-English meaning | How to read values |
|---|---|---|
| `target_dropout_rate` / **Abandonment rate (%)** | Official INEP *taxa de abandono*: share of students who **stopped attending** during the school year after the census reference date | 0 = nobody left; 5 = 5% left. Higher = worse |
| `year` | School census / rendimento year | Years present in the mart (currently 2018–2025) |
| `school_id` (`CO_ENTIDADE`) | Unique school code (INEP) | Join key between Censo and Rendimento |
| `uf` | Brazilian state abbreviation | e.g. SP, BA, AM |
| `municipio_id` | IBGE municipality code | Geographic context |
| `tp_dependencia` | Administrative network | 1=Federal, 2=State, 3=Municipal, 4=Private |
| `tp_localizacao` | School location type | 1=Urban, 2=Rural |
| `is_rural` | 1 if rural school | Shortcut of `tp_localizacao == 2` |
| `is_public` | 1 if public network (federal/state/municipal) | 0 = private |
| `in_agua` | Has potable / public water | 1=yes, 0=no |
| `in_energia` | Connected to public electricity | 1=yes, 0=no |
| `in_esgoto` | Public sewage connection | 1=yes, 0=no |
| `in_internet` | Internet available at school | 1=yes, 0=no |
| `in_biblioteca` | Library / reading room | 1=yes, 0=no |
| `in_lab_info` | Computer lab | 1=yes, 0=no |
| `in_quadra` | Sports court | 1=yes, 0=no |
| `qt_mat_bas` | Enrollment count (basic education total, when available) | Larger = bigger school |
| `enrollment_level` | Enrollment used for this level (Fundamental or Médio) | Filter requires ≥ 20 students |
| `qt_doc_bas` | Number of teachers (basic education) | Staffing intensity |
| `student_teacher_ratio` | Students ÷ teachers | Higher often means more crowded classes |
| `risk_band` | low / moderate / high | Relative triage label inside the level |
| `high_risk` | 1 if school is in the elevated-risk group | Binary flag for triage demos |
| **MAE / RMSE / R²** | Model error metrics (later notebooks / model folder) | Lower MAE/RMSE better; R² closer to 1 better |

### Acronyms
| Acronym | Meaning |
|---|---|
| **INEP** | Brazilian federal education statistics institute |
| **Censo Escolar** | Annual school census (structure + enrollment) |
| **Taxas de Rendimento** | Official approval / failure / abandonment rates |
| **EDA** | Exploratory Data Analysis |
| **UF** | Federative unit (state) |
| **Fundamental** | Ensino Fundamental (approx. primary + lower secondary) |
| **Médio** | Ensino Médio (upper secondary) |


In [ ]:
# Cell A — project path only (no Path.cwd / exists / resolve)
import sys
ROOT = r"C:\Users\User\Desktop\Projeto Living Stone Foundation"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


### What this cell did
Added the project folder to Python’s import path using a **fixed string** (no `Path.cwd()` / `exists` / `resolve`). Those path checks can freeze kernels on Desktop/OneDrive.


In [ ]:
# Cell B — imports (first run can take a minute for pandas/seaborn)
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FEATURE_LABELS = {
    "is_rural": "Rural school (1=yes)",
    "is_public": "Public network (1=yes)",
    "in_internet": "Has internet (1=yes)",
    "in_lab_info": "Has computer lab (1=yes)",
    "in_quadra": "Has sports court (1=yes)",
    "in_biblioteca": "Has library (1=yes)",
    "in_agua": "Has water (1=yes)",
    "in_energia": "Has electricity (1=yes)",
    "in_esgoto": "Has sewage (1=yes)",
    "enrollment_level": "Enrollment (this level)",
    "qt_mat_bas": "Basic-ed enrollment (total)",
    "qt_doc_bas": "Number of teachers",
    "student_teacher_ratio": "Students per teacher",
    "tp_dependencia": "Admin network code",
    "tp_localizacao": "Urban/rural code",
    "target_dropout_rate": "Abandonment rate (%)",
}
from src.eda import load_level_mart, missingness_table, summary_by_uf
print("Imports OK")


### What this cell did
Loaded charting/table libraries. The **first** run can take ~30–90s while pandas/seaborn warm up — that is normal, not a freeze on `import sys`.


In [ ]:
df = load_level_mart('fundamental')
print('Rows (school x year):', df.shape[0])
print('Columns:', df.shape[1])
print('Unique schools:', df['school_id'].nunique())
print('Years:', sorted(df['year'].dropna().astype(int).unique().tolist()))
display(df.head())


### How to read this preview
- Each **row** = one school in one year for **Ensino Fundamental**.
- `target_dropout_rate` is the official abandonment % we will try to predict later.
- Feature columns (`in_*`, `is_*`, enrollment, teachers) come from the **Censo Escolar**.

### Insight
You are looking at a **school risk table**, not a student list. Interventions suggested by this project are about **which schools to prioritize**, not which child to call.


## 1. Distribution of the abandonment rate


In [ ]:
display(df['target_dropout_rate'].describe().rename('Abandonment rate (%)'))
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['target_dropout_rate'], bins=40, ax=ax, color='#1f4e79')
ax.set_title('How common is each abandonment rate?')
ax.set_xlabel('Official abandonment rate (%)')
ax.set_ylabel('Number of school-year rows')
plt.tight_layout()
plt.show()
print('Share of rows with 0% abandonment:', float((df['target_dropout_rate'] == 0).mean()))
print('Share of rows with abandonment ≥ 5%:', float((df['target_dropout_rate'] >= 5).mean()))


### How to read this chart
- **X-axis:** abandonment rate in percent (0–100 scale, usually concentrated near 0).
- **Y-axis:** how many school-year observations fall in that bin.
- A tall bar at 0% means many schools reported **no abandonment** that year.

### Insight for Ensino Fundamental
Abandonment is a **rare / skewed** school outcome. That is normal. Practical consequences:
1. Averages can look “small” while a minority of schools is still in serious trouble (look at the right tail / 90th percentile).
2. Model quality should be judged with **MAE/RMSE**, not classification accuracy.
3. Triage should focus on the **high tail**, not on shaming schools with 0–1%.


## 2. Data quality (missing values)


In [ ]:
miss = missingness_table(df).head(15).rename(columns={
    'pct_missing': 'Share missing',
    'n_missing': 'Count missing',
    'dtype': 'Data type',
})
display(miss)


### How to read this table
- **Share missing = 0.000** means the column is fully filled in this mart.
- Columns near the top are the emptiest (if any).

### Insight
For modeling we already kept rows with a non-null official abandonment rate. If a *feature* is heavily missing, either drop it or impute carefully — never silently treat blanks as zeros without saying so.


## 3. Where is abandonment higher? (by state / UF)


In [ ]:
g = summary_by_uf(df, min_n=30).rename(columns={
    'uf': 'State (UF)',
    'n': 'School-year rows',
    'mean_abandono': 'Mean abandonment (%)',
    'median_abandono': 'Median abandonment (%)',
})
display(g.head(15))
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=g.head(12), x='State (UF)', y='Mean abandonment (%)', ax=ax, color='#9c2a2a')
ax.set_title('States with highest mean school abandonment (only UF with ≥ 30 rows)')
ax.set_xlabel('State (UF)')
ax.set_ylabel('Mean abandonment rate (%)')
plt.tight_layout()
plt.show()


### How to read this chart
- Each bar is a **state (UF)**.
- Height = average school abandonment rate among schools of **Ensino Fundamental** in that state (with enough sample size).
- We require **n ≥ 30** so tiny samples do not create fake “worst state” headlines.

### Insight
Use this as a **map of attention**, not as a final ranking for newspapers:
- Different states have different networks (more municipal vs state schools).
- Coverage and school size differ.
- Still, persistently high bars suggest where network managers may want deeper local diagnostics.


## 4. Which school traits move with abandonment? (simple correlations)


In [ ]:
feat = ['is_rural','is_public','in_internet','in_lab_info','in_quadra','in_biblioteca','enrollment_level','qt_doc_bas']
feat = [c for c in feat if c in df.columns]
tmp = df[feat + ['target_dropout_rate']].copy()
if 'qt_doc_bas' in tmp.columns and 'enrollment_level' in tmp.columns:
    tmp['student_teacher_ratio'] = tmp['enrollment_level'] / tmp['qt_doc_bas'].replace({0: pd.NA})
    feat = feat + ['student_teacher_ratio']
corr = tmp[feat + ['target_dropout_rate']].corr(numeric_only=True)['target_dropout_rate'].drop('target_dropout_rate')
corr_named = corr.rename(index=FEATURE_LABELS).sort_values()
display(corr_named.rename('Correlation with abandonment rate').to_frame())
ax = corr_named.plot(kind='barh', figsize=(9, 5), color='#2a9d8f')
ax.set_title('Simple one-variable associations with abandonment')
ax.set_xlabel('Correlation (−1 to +1). Positive = higher feature ↔ higher abandonment')
plt.tight_layout()
plt.show()


### How to read this chart
- Each bar is **one school characteristic**.
- Axis goes from −1 to +1:
  - **Positive** correlation: when the feature is higher/true, abandonment tends to be **higher**.
  - **Negative** correlation: when the feature is higher/true, abandonment tends to be **lower**.
- Example readings:
  - `Rural school (1=yes)` positive ⇒ rural schools show higher abandonment **on average** in this table.
  - `Has internet (1=yes)` negative ⇒ schools with internet show lower abandonment **on average**.

### Critical caveats (please keep)
1. Correlation is **not causation** (internet does not automatically “cause” retention).
2. This is **univariate** (one variable at a time). The ML model can reorder importance when variables are combined.
3. Compare later with `models/fundamental/figures/global_importance_top.csv`.

### Insight for Ensino Fundamental
Write down the 3 strongest bars (positive and negative). Those become the plain-language story for counselors/managers in the Streamlit app for this level — and they may **differ** from the other education level (see notebook 04).


## 5. Optional: compare with the trained model drivers


In [ ]:
imp_path = Path(ROOT) / 'models' / 'fundamental' / 'figures' / 'global_importance_top.csv'
try:
    imp = pd.read_csv(imp_path)
    imp['feature_label'] = imp['feature'].map(FEATURE_LABELS).fillna(imp['feature'])
    display(imp[['feature_label','importance']].head(12))
    ax = imp.head(10).set_index('feature_label')['importance'].plot(kind='barh', color='#1f4e79', figsize=(9,4))
    ax.invert_yaxis()
    ax.set_title('Model feature importance (trained fundamental model)')
    ax.set_xlabel('Importance (higher = model relies on this more)')
    plt.tight_layout(); plt.show()
except FileNotFoundError:
    print('Train first: python -m src.train --level fundamental')


### How to read model importance
Unlike correlation, this answers: **“When the trained model predicts abandonment, which inputs move the prediction most?”**

### Insight
If correlation and model importance **agree** (e.g. rurality / public network / staffing), the narrative is robust.  
If they **disagree**, prefer the model list for triage explanations, but still show both transparently — disagreement often means confounding (variables overlapping).

### Closing takeaway (Ensino Fundamental)
1. Outcome = official school abandonment %, skewed toward zero.  
2. Geography highlights uneven risk across states.  
3. Structural/staffing traits associate with risk — useful for prioritization, not blame.  
4. This notebook alone does **not** authorize student-level interventions.
